In [7]:
import random
from scripts.make_mastertables import count_no_of_gaps, find_DEL_closest_to_cut, find_min_and_max_gap_ranges, find_DEL_around_cut_range, size_of_del_around_the_cut, gap_span_analysed_window, return_shifted_DEL_range, return_category_of_DEL, return_cut_type_based_on_cut_categories, find_microhomology
import pandas as pd


In [8]:
# This function generates a random nucleotide sequence containing A, T, C and G letters in a lenght of 100 charecters. In the sequence radomly occur deletions represented as '-', those deletions are 0-100 charecters long and replacing A, T, C and G characters. Number of such deletions is 0-2.

def random_nt_seq_generator():
    """
    Generates a random nucleotide sequence of length 100 with deletions represented as '-'.
    The deletions are randomly placed in the sequence and can be of variable length.
    In the middle od the sequence is predicted 'cut_site' = 50 followed by PAM -> two 'G' nucleotides are at positions 54 and 55.
    """

    # Generate a random nucleotide sequence of length 100
    sequence_length = 100
    nucleotides = ['A', 'T', 'C', 'G']
    Reference_Sequence = ''.join(random.choices(nucleotides, k=sequence_length))
    # Set position  52 and 53 as 'G'
    Reference_Sequence = Reference_Sequence[:54] + 'GG' + Reference_Sequence[56:]

    start_pos = random.randint(0, sequence_length-1)
    end_pos = random.randint(start_pos, sequence_length-1)

    # Replace the selected range with '-'
    Aligned_Sequence = Reference_Sequence[:start_pos] + '-' * (end_pos - start_pos) + Reference_Sequence[end_pos:]

    return Aligned_Sequence, Reference_Sequence

def generate_random_sequences_to_file(filename, num_sequences=100):
    with open(filename, 'w') as file:
        file.write("Aligned_Sequence\tReference_Sequence\n")
        for _ in range(num_sequences):
            Aligned_Sequence, Reference_Sequence = random_nt_seq_generator()
            file.write(Aligned_Sequence + '\t' + Reference_Sequence+ '\n')

def unedited(aligned_sequence, reference_sequence):
    if aligned_sequence == reference_sequence:
        return True
    else:
        return False

def apply_find_microhomology(df, func, new_column_names):
    """Adds new columns into table based on the function find_microhomology."""
    results = df.apply(lambda row: func(row['Gap_closest_to_cut_range'][0],
                                        row['Gap_closest_to_cut_range'][1],
                                        row['Gap_span_analysed_window'],
                                        row['Aligned_Sequence'],
                                        row['Reference_Sequence']), axis=1)
    # Unpack results into separate columns
    for i, col_name in enumerate(new_column_names):
        df[col_name] = [result[i] for result in results]


In [17]:
# Generate a random nucleotide sequence with deletions into a csv file for 100 sequences and name them as 'Reference_Sequence' and 'Aligned_Sequence'

output_filename = "../../CRISPRlysis_outputs/random/random_sequences.tsv"
num_sequences = 200000 #1000000 7 seconds

generate_random_sequences_to_file(output_filename, num_sequences=num_sequences)
print(f"Generated {num_sequences} random nucleotide sequences with deletions in '{output_filename}'")


Generated 200000 random nucleotide sequences with deletions in '../../CRISPRlysis_outputs/random/random_sequences.tsv'


In [18]:
# 1000000 15 seconds
df = pd.read_csv(output_filename, sep='\t')

# Add columns to the dataframe as it would be the CRIPSResso2 output
df['Unedited'] = df.apply(lambda row: unedited(row['Aligned_Sequence'], row['Reference_Sequence']), axis=1)
df['n_deleted'] = df.apply(lambda row: row['Aligned_Sequence'].count('-'), axis=1)
df['n_inserted'] = 0
df['n_mutated'] = 0
df['#Reads'] = 1
df['%Reads'] = 1/df.shape[0]
df['gRNA_seq'] = df.apply(lambda row: row['Reference_Sequence'][33:53], axis=1)
df['sample'] = 'random_1_SpCas9_none_1_none_1_0_1'


# Save dataframe to a new file
output_filename = "../../CRISPRlysis_outputs/random/random_sequences_CRISPResso2_like.tsv"
df.to_csv(output_filename, sep='\t', index=False)
print(f"Saved the dataframe with CRISPResso2-like columns to '{output_filename}'")
df.shape

Saved the dataframe with CRISPResso2-like columns to '../../CRISPRlysis_outputs/random/random_sequences_CRISPResso2_like.tsv'


(200000, 10)